# CDC benchmark report

Reads the JSON files written to `results/` by `load_update.py`,
`measure_lag.py` (probe / report) and the pgbench comparison, and produces the
tables and figures for `docs/results.md`.

Run from the repo root venv: `pip install -r benchmark/requirements.txt jupyter`.

In [ ]:
import json, re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULTS = Path("..") / "results" if Path("..", "results").exists() else Path("results")
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"   # reference palette slots 1-3
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GRID,
    "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "font.size": 10,
})

def load(kind):
    rows = []
    for p in sorted(RESULTS.glob(f"*-{kind}*.json")):
        d = json.loads(p.read_text(encoding="utf-8"))
        d["file"] = p.name
        rows.append(d)
    return rows

loads, probes, reports = load("load"), load("probe"), load("report")
print(f"{len(loads)} load runs, {len(probes)} probe runs, {len(reports)} reports in {RESULTS.resolve()}")

## Load runs (PostgreSQL side)

In [ ]:
pd.DataFrame([{ "file": d["file"], "mode": d["mode"], "entity": d["entity"], "updates": d["updates"],
                "elapsed_s": round(d["elapsed_s"], 2), "pg_rate_per_s": round(d["pg_rate_per_s"]) }
              for d in loads])

## Commit → searchable (probe)

Measured from outside the pipeline: update committed in PostgreSQL → visible to
`_search` (includes the 1 s `refresh_interval`). Empirical CDF per run, with
p50 / p95 / p99 marked on the latest run.

In [ ]:
if probes:
    fig, ax = plt.subplots(figsize=(8, 4.2))
    for i, d in enumerate(probes[-3:]):                      # at most 3 runs => 3 fixed colours
        s = pd.Series(sorted(d["samples_ms"]))
        y = (s.index + 1) / len(s)
        ax.step(s.values, y, where="post", lw=2, color=[BLUE, ORANGE, AQUA][i], label=d["file"])
    last = probes[-1]["summary_ms"]
    for q in ("p50", "p95", "p99"):
        ax.axvline(last[q], color=MUTED, lw=1, ls="--")
        ax.annotate(f"{q} {last[q]:.0f} ms", (last[q], 0.03), xytext=(4, 0), textcoords="offset points",
                    color=INK, fontsize=9, rotation=90, va="bottom", ha="left")
    ax.set_axisbelow(True)
    ax.set_xlabel("commit → searchable (ms)")
    ax.set_ylabel("share of probes")
    ax.set_ylim(0, 1.02)
    if len(probes[-3:]) > 1:
        ax.legend(frameon=False, loc="lower right")
    ax.set_title("Commit-to-searchable latency (ECDF)", loc="left", color=INK)
    plt.tight_layout(); plt.show()

pd.DataFrame([{ "file": d["file"], **{k: round(v) for k, v in d["summary_ms"].items()}, "timeouts": d["timeouts"] }
              for d in probes])

## Lag per pipeline stage (from `cdc.*` timestamps)

`report` samples the latest version of each document touched in the window.
p95 per stage, one row per report.

In [ ]:
rows = []
for d in reports:
    for stage, s in d["stages_ms"].items():
        rows.append({"report": d["file"], "stage": stage, **{k: s[k] for k in ("count", "p50", "p95", "p99")}})
stages = pd.DataFrame(rows)
stages.pivot_table(index="report", columns="stage", values="p95").round(0) if not stages.empty else stages

In [ ]:
if reports:
    d = reports[-1]
    parts = {k: v["p95"] for k, v in d["stages_ms"].items() if not k.startswith("e2e")}
    fig, ax = plt.subplots(figsize=(8, 3.2))
    names, vals = list(parts), list(parts.values())
    pos = list(range(len(names)))
    ax.barh(pos, vals, color=BLUE, height=0.55)
    for y, v in zip(pos, vals):
        ax.text(v, y, f" {v:.0f} ms", va="center", color=INK, fontsize=9)
    ax.set_yticks(pos, names)
    ax.set_axisbelow(True); ax.margins(x=0.12)
    ax.invert_yaxis(); ax.grid(axis="y", visible=False)
    ax.set_xlabel("p95 (ms)")
    ax.set_title(f"p95 lag by stage — {d['file']}", loc="left", color=INK)
    plt.tight_layout(); plt.show()

## pgbench: does CDC slow the primary down?

Parses `results/*-pgbench/{baseline,cdc,paused}.txt` written by
`benchmark/pgbench/compare.sh`.

In [ ]:
def parse_pgbench(path):
    txt = path.read_text(encoding="utf-8", errors="replace")
    tps = re.search(r"tps = ([\d.]+)", txt)
    lat = re.search(r"latency average = ([\d.]+) ms", txt)
    return {"tps": float(tps.group(1)) if tps else None, "latency_ms": float(lat.group(1)) if lat else None}

bench = []
for run in sorted(RESULTS.glob("*-pgbench")):
    for label in ("baseline", "cdc", "paused"):
        f = run / f"{label}.txt"
        if f.exists():
            bench.append({"run": run.name, "setup": label, **parse_pgbench(f)})
bench = pd.DataFrame(bench)
if not bench.empty:
    base = bench[bench.setup == "baseline"].set_index("run")["tps"]
    bench["tps_vs_baseline_%"] = bench.apply(lambda r: round(100 * (r.tps / base.get(r.run, float("nan")) - 1), 1), axis=1)
bench